# 03 · Đánh giá

1. Một lượt chạy trên train (có đáp án): điểm, theo loại / shop / dạng câu, nhóm lỗi, ngưỡng bỏ phiếu và oracle@K.
2. Các bản nộp public: so với nhau, không cần nhãn (điểm public chỉ có ở mức tổng, không bao giờ làm nhãn cho từng câu).

In [ ]:
# Setup. Local checkout: nothing to install. Kaggle: Internet on, GPU accelerator for inference / training, the competition data
# attached as a private dataset, and the code either attached as a dataset (a copy of the checkout) or cloned with your
# GitHub token in Add-ons -> Secrets as GITHUB_TOKEN (if the repo is private). Colab: Google Drive holding MyDrive/datamate/code and
# MyDrive/datamate/data (or a GITHUB_TOKEN secret for the code), a GPU runtime for inference / training.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/hana200804/datamate-agent-Locked-In1.git"


def clone(root, token):
    subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{token}@{REPO_URL}", str(root)], check=True)
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=root, check=True)  # token out of saved files


if Path("/kaggle/input").exists():
    root = Path("/tmp/datamate-agent-Locked-In1")
    attached = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/datamate/notebook.py"))
    if not root.exists() and attached:
        shutil.copytree(attached[0], root)                         # code attached as a dataset: no token needed
    if not root.exists():
        from kaggle_secrets import UserSecretsClient
        clone(root, UserSecretsClient().get_secret("GITHUB_TOKEN"))
elif "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    root, code = Path("/content/datamate-agent-Locked-In1"), Path("/content/drive/MyDrive/datamate/code")
    if not root.exists() and (code / "datamate/notebook.py").exists():
        shutil.copytree(code, root)                                # code uploaded to Drive: no token needed
    if not root.exists():
        from google.colab import userdata                          # key icon -> secret GITHUB_TOKEN with notebook access
        clone(root, userdata.get("GITHUB_TOKEN"))
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("base")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
RUN = "runs/nb_train"     # a train run (its trace.jsonl, or shard*/trace.jsonl), e.g. runs/nb_train from 02_inference
SUBS = {"base": "runs/public_base", "lora": "runs/public_lora"}   # public submissions to compare

In [ ]:
from datamate import scoring, voting
from datamate.hints import DEBT_WORDS
from datamate.ingest import norm
run = env.root / RUN if not str(RUN).startswith("/") else Path(RUN)
gold = {a["id"]: a for a in read(env.data / "train/answers.jsonl")}
trace = {r["id"]: r for p in ([run / "trace.jsonl"] if (run / "trace.jsonl").exists() else sorted(run.glob("shard*/trace.jsonl"))) for r in read(p)}
print(len(trace), "traced questions")

## 1. Điểm của lượt chạy train

`scoring.evaluate` cho PA, độ chính xác khi trả lời, khoảng tin cậy bootstrap theo shop và phân rã theo loại đáp án.

In [ ]:
questions = {q["id"]: q for q in read(env.data / "train/questions.jsonl")}
report, rows = scoring.evaluate({i: gold[i] for i in trace}, {i: r["final"] for i, r in trace.items()}, questions)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in report.items() if k in ("N", "correct", "wrong", "abstain", "PA", "precision", "coverage")})
print("PA 95% CI (shop bootstrap):", [round(x, 3) for x in report["PA_cluster_bootstrap_95"]])
pd.DataFrame(report["breakdowns"]["gold_type"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("N", ascending=False)

In [ ]:
by_shop = pd.DataFrame(report["breakdowns"]["business_id"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("PA")
by_shop.plot.barh(y="PA", figsize=(6, 7), legend=False, title="PA by shop")
by_shop.head(8)

## Nhóm lỗi

Mỗi câu chưa đúng thuộc một nhóm: không mẫu nào đúng (cần model / prompt tốt hơn), có mẫu đúng nhưng nhóm sai thắng phiếu, hoặc nhóm đúng
đứng đầu nhưng dưới ngưỡng nên bỏ trống (cần bỏ phiếu / ngưỡng tốt hơn).

In [ ]:
def kind(r):
    g = gold[r["id"]]
    s = scoring.score(r["final"], g)
    if s == 1: return "right"
    if not any(p and scoring.score(p, g) == 1 for p in (x["pred"] for x in r["samples"])): return "no right sample"
    if s == -1: return "wrong: a wrong group won"
    top_right = bool(r.get("vote")) and scoring.score(r["vote"], g) == 1
    return "abstain: right group on top, below threshold" if top_right else "abstain: right sample, another group on top"
pd.Series(Counter(kind(r) for r in trace.values())).sort_values(ascending=False).to_frame("questions")

## Ngưỡng bỏ phiếu và oracle@K

PA khi đổi ngưỡng đồng thuận (bỏ phiếu lại trên các mẫu đã lưu), và tỷ lệ câu có ít nhất một mẫu đúng (trần của việc chọn đáp án).

In [ ]:
def replay(r, tau):
    texts, preds = [s["text"] for s in r["samples"]], [s["pred"] for s in r["samples"]]
    pool = voting.debt_pool(texts, 8, bool(DEBT_WORDS.search(norm(r["question"]))))
    ans, conf = voting.vote([preds[j] for j in pool], len(pool))
    return voting.final_answer(ans, conf, tau, r["final"].get("reason") == "period outside the shop's data dates", True)
sweep = pd.Series({t: sum(scoring.score(replay(r, t), gold[i]) for i, r in trace.items()) / len(trace) for t in (0.25, 0.375, 0.5, 0.625, 0.75)}, name="PA")
oracle = sum(any(p and scoring.score(p, gold[i]) == 1 for p in (s["pred"] for s in r["samples"])) for i, r in trace.items()) / len(trace)
print(f"oracle (any sample right): {oracle:.3f}")
sweep.to_frame()

## Theo dạng câu hỏi

In [ ]:
from datamate import hints as H
shape = lambda q: ("debt" if DEBT_WORDS.search(norm(q)) else "profit" if H.PROFIT.search(norm(q)) else "trips" if H.TRIPS.search(norm(q))
                   else "lots" if H.LOT.search(norm(q)) else "other")
df = pd.DataFrame([{"shape": shape(r["question"]), "score": scoring.score(r["final"], gold[i])} for i, r in trace.items()])
df.groupby("shape")["score"].agg(n="size", PA="mean").sort_values("n", ascending=False).round(3)

## 2. So các bản nộp public (không nhãn)

Số câu trả lời, mức trùng đáp án giữa các file, và chỗ khác nhau theo shop giữa hai bản cuối.

In [ ]:
def key(a):
    t, x = a.get("type"), a.get("answer")
    if t == "abstain" or (x is None and t != "unanswerable"): return "ABSTAIN"
    return "UNANSWERABLE" if t == "unanswerable" else json.dumps([t, round(x, 2) if isinstance(x, float) else x], ensure_ascii=False, sort_keys=True)
pq = {q["id"]: q for q in read(env.data / "public_test/questions.jsonl")}
subs = {name: {r["id"]: key(r) for r in read(env.root / p / "submission.jsonl")} for name, p in SUBS.items() if (env.root / p / "submission.jsonl").exists()}
pd.DataFrame({n: {"answered": sum(v != "ABSTAIN" for v in s.values()), "unanswerable": sum(v == "UNANSWERABLE" for v in s.values())} for n, s in subs.items()}).T

In [ ]:
names = list(subs)
pd.DataFrame([[sum(subs[a][i] == subs[b][i] for i in pq) / len(pq) for b in names] for a in names], index=names, columns=names).style.format("{:.1%}")

In [ ]:
if len(names) >= 2:
    a, b = names[-2], names[-1]
    display(pd.DataFrame([{"shop": q["business_id"], "changed": subs[a][i] != subs[b][i], "newly answered": subs[a][i] == "ABSTAIN" != subs[b][i],
                           "newly abstained": subs[b][i] == "ABSTAIN" != subs[a][i]} for i, q in pq.items()]).groupby("shop").sum())